# X-VLA Colab Server
Serves the `lerobot/xvla-widowx` policy via FastAPI + ngrok.
Run all cells, then paste the ngrok URL into your local client.

In [ ]:
# Install dependencies
!pip install -q fastapi uvicorn pyngrok lerobot transformers pillow

In [ ]:
import torch
from transformers import AutoTokenizer
from lerobot.policies.xvla.modeling_xvla import XVLAPolicy

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

policy = XVLAPolicy.from_pretrained("lerobot/xvla-widowx").to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(policy.config.tokenizer_name)
print("Model loaded.")

In [ ]:
import base64
import io
import numpy as np
from PIL import Image
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List

app = FastAPI()

# Cache tokenized instructions to avoid re-tokenizing every call
_token_cache = {}

def decode_image(b64: str) -> torch.Tensor:
    """Base64 PNG -> (1, 3, H, W) float tensor on device."""
    img = Image.open(io.BytesIO(base64.b64decode(b64))).convert("RGB")
    arr = np.array(img, dtype=np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(device)

def get_language_tokens(task: str):
    if task not in _token_cache:
        tok = tokenizer(
            task,
            padding="max_length",
            max_length=policy.config.tokenizer_max_length,
            truncation=True,
            return_tensors="pt",
        )
        _token_cache[task] = (
            tok["input_ids"].to(device),
            tok["attention_mask"].to(device),
        )
    return _token_cache[task]


class InferenceRequest(BaseModel):
    image_up: str          # base64 PNG
    image_side: str        # base64 PNG
    robot_qpos: List[float]  # 6 joint positions
    task: str


@app.post("/predict")
def predict(req: InferenceRequest):
    language_tokens, language_attention_mask = get_language_tokens(req.task)

    observation = {
        "observation.images.image":  decode_image(req.image_up),
        "observation.images.image2": decode_image(req.image_side),
        "observation.state": torch.tensor(req.robot_qpos, dtype=torch.float32).unsqueeze(0).to(device),
        "observation.language.tokens": language_tokens,
        "observation.language.attention_mask": language_attention_mask,
    }

    with torch.inference_mode():
        actions = policy.select_action(observation)

    actions_np = actions.detach().cpu().numpy().flatten().tolist()
    return {"actions": actions_np}


@app.get("/health")
def health():
    return {"status": "ok", "device": device}


print("FastAPI app defined.")

In [ ]:
from pyngrok import ngrok
import uvicorn
import threading

# Paste your ngrok auth token here (free at https://dashboard.ngrok.com)
NGROK_AUTH_TOKEN = ""  # <-- fill in

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

PORT = 8000
tunnel = ngrok.connect(PORT)
print(f"\n Public URL: {tunnel.public_url}")
print(f" POST {tunnel.public_url}/predict")

# Run uvicorn in a background thread so the cell doesn't block
def run():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

thread = threading.Thread(target=run, daemon=True)
thread.start()